# Notebook 04 — Workflow Modelling & Metrics
## Class 7 Skill: Represent entities, events/states, interactions; build relational model; calculate 3–5 metrics

This notebook covers:
- The trip lifecycle as an event/state model
- Entity-relationship structure (Trip, Zone, Vendor)
- Joins and aggregations to build the enriched trip table
- Calculation and interpretation of all 5 business metrics

> FDE mindset: We are not just analysing data — we are modelling a real-world workflow and mapping metrics to it.

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd

from ingest import ingest_trips, ingest_zones, ingest_vendors
from validate import validate
from transform import build_enriched_trips, calculate_metrics, print_metrics_summary

print('Loading and validating data...')
trips_df   = ingest_trips()
zones_df   = ingest_zones()
vendors_df = ingest_vendors()
valid_df, _ = validate(trips_df)
print(f'Using {len(valid_df):,} validated trips.')

## 1. Trip Lifecycle — Event/State Model

```
[Trip Requested] → [Pickup Recorded] → [Trip In Progress] → [Dropoff Recorded]
                                                                    ↓
                                                          [Fare Calculated]
                                                                    ↓
                                                          [Payment Processed]
                                                                    ↓
                                                          [Record Written]
                                                                    ↓
                                                          [Validation Gate]
                                                                   / \
                                                              VALID    REJECTED
                                                                ↓
                                                          [Metric Calculation]
```

Each trip goes through this sequence. The **Validation Gate** is the FDE's intervention — only trips that pass all 8 rules contribute to KPIs.

## 2. Entity Model

Three entities, two dimensions joining to one fact table:

```
Vendor ──────────────── Trip ──────────────── Zone
(VendorID PK)          (VendorID FK)          (LocationID PK)
(vendor_name)          (pickup_datetime)      (zone_name)
                       (dropoff_datetime)     (borough)
                       (PULocationID FK)      (service_zone)
                       (DOLocationID FK)
                       (fare_amount)
                       (trip_distance)
                       (duration_minutes)     ← derived
                       (avg_speed_mph)        ← derived
                       (total_revenue)        ← derived
```

In [ ]:
# Build the enriched (joined) trip table
print('Building enriched trip table (joins: trips + zones + vendors) ...')
enriched_df = build_enriched_trips(valid_df, zones_df, vendors_df)

print(f'\nEnriched table shape: {enriched_df.shape}')
print(f'Columns added by joins: pickup_borough, pickup_zone, dropoff_borough, dropoff_zone, vendor_name')
print(f'Derived columns: duration_minutes, avg_speed_mph, fare_per_mile, total_revenue, pickup_hour')
print('\nSample row:')
display_cols = ['vendor_name', 'pickup_borough', 'pickup_zone', 'duration_minutes',
                'fare_amount', 'tip_amount', 'total_revenue', 'avg_speed_mph']
print(enriched_df[display_cols].head(3).to_string(index=False))

## 3. Calculating All 5 Metrics

In [ ]:
print('Calculating all 5 business metrics ...')
metrics = calculate_metrics(enriched_df)

### M1 — Revenue per Pickup Borough
**KPI link:** This is the primary KPI. Tells the Operations Director which boroughs drive revenue.

In [ ]:
m1 = metrics['M1_Revenue_per_Borough']
print('M1 — Revenue by Pickup Borough:')
print(m1.to_string(index=False))

### M2 — Average Trip Duration by Pickup Hour
**KPI link:** Operational anomaly detection — rush hours should have longer trips; overnight should be shorter.

In [ ]:
m2 = metrics['M2_Avg_Duration_by_Hour']
print('M2 — Avg Trip Duration by Hour (all 24 hours):')
print(m2.to_string(index=False))

### M3 — Trip Count per Vendor
**KPI link:** Data completeness check — if one vendor is far below expected, they may have a reporting gap.

In [ ]:
m3 = metrics['M3_Trip_Count_per_Vendor']
print('M3 — Trip Count per Vendor:')
print(m3.to_string(index=False))

### M4 — Speed Plausibility Rate
**KPI link:** Data trustworthiness signal — a low plausibility rate means the validated dataset has underlying GPS/meter issues.

In [ ]:
m4 = metrics['M4_Speed_Plausibility_Rate']
print('M4 — Speed Plausibility Rate:')
print(m4.to_string(index=False))

### M5 — Median Fare per Mile by Borough
**KPI link:** Pricing consistency — significant differences between boroughs warrant investigation.

In [ ]:
m5 = metrics['M5_Median_Fare_per_Mile_by_Borough']
print('M5 — Median Fare per Mile by Borough:')
print(m5.to_string(index=False))

## 4. Full Metrics Summary

In [ ]:
print_metrics_summary(metrics)

## 5. Metric Interpretations

| Metric | What it tells us | Action it enables |
|---|---|---|
| M1 | Manhattan generates the most revenue; EWR (airport) generates high revenue per trip | Prioritise dispatch to Manhattan during peak hours |
| M2 | Evening hours (5–8 PM) have the longest average trips — consistent with rush-hour traffic | Alert operators if Oct/Nov shows a different pattern (possible data issue) |
| M3 | VeriFone (Vendor 2) typically has more trips than CMT (Vendor 1) — if this reverses, investigate | Trigger vendor compliance review |
| M4 | A plausibility rate below 95% is a warning sign | Flag the month for data quality review before publishing KPIs |
| M5 | Airport zones tend to have a lower fare-per-mile (flat-rate airport fares) | Correct for this when comparing zone revenue per mile |